# SL-10 --- Capstone : un pipeline neuro-symbolique de bout en bout

**Navigation** : [Index](./README.md) | [<< SL-9](SL-9-InverseResolution.ipynb)

## Objectifs d'apprentissage

Ce notebook integre les briques de toute la serie en **un seul pipeline** qui part
de texte brut et finit par decouvrir des faits que le texte ne dit pas :

1. **Extraire** des faits structures d'un corpus avec un **vrai LLM** (SL-7)
2. **Valider** ces faits par un oracle symbolique : schema type, contraintes fonctionnelles (SL-1, SL-3)
3. **Peupler** un graphe de connaissances (SL-6)
4. **Miner** des regles de Horn par support/confiance, facon AMIE (SL-4, SL-6, SL-9)
5. **Inferer** des faits nouveaux par chainage avant, avec provenance (SL-2)
6. **Confronter** la reponse du LLM seul a celle du KG raisonne (SL-5, SL-7)

### Prerequis
- La serie SL-1 a SL-9 (chaque etage renvoie au notebook qui l'a introduit)
- Acces LLM optionnel : `.env` a la racine de la serie (cf SL-7) ; un simulateur
  deterministe prend le relais hors-ligne

### Duree estimee : 90 minutes


---

## 1. Le pipeline

```
   corpus (texte brut)
        |
        v
  [1] EXTRACTION LLM  ............ neuronal, faillible        (SL-7)
        |  triplets candidats
        v
  [2] ORACLE SYMBOLIQUE .......... types + contraintes        (SL-1, SL-3)
        |  triplets valides
        v
  [3] GRAPHE DE CONNAISSANCES .... stockage relationnel       (SL-6)
        |
        v
  [4] MINING DE REGLES ........... support / confiance        (SL-4, SL-6, SL-9)
        |  regles de Horn scorees
        v
  [5] CHAINAGE AVANT ............. faits NOUVEAUX + provenance (SL-2)
        |
        v
  [6] CONFRONTATION .............. LLM seul vs KG raisonne    (SL-5, SL-7)
```

La these de la serie, version operationnelle : **le neuronal propose, le
symbolique dispose**. Le LLM lit le texte mieux qu'aucun parseur ; le symbolique
verifie, structure et raisonne mieux qu'aucun LLM. Chaque etage va exhiber a la
fois sa force et son angle mort --- y compris le symbolique.


---

## 2. Le corpus et le schema

Une saga familiale et industrielle inventee. Notez les trois derniers enonces :
deux sont des pieges deliberes (un registre contradictoire, une note mal typee),
le troisieme est une information que seul le *raisonnement* permettra d'exploiter.


In [1]:
CORPUS = """
Henri Verne a fonde l'Atelier Verne en 1952, a Lyon.
Henri a deux enfants : Claire et Marc.
Claire dirige aujourd'hui l'Atelier Verne, ou elle travaille depuis trente ans.
Marc travaille lui aussi pour l'Atelier Verne.
Sophie est la fille de Claire, et elle travaille pour l'Atelier Verne.
Lucas est le fils de Marc.
Henri est le grand-pere de Lucas.
Nadia, la mere de Lucas, travaille pour la Fonderie Brun.
Paul dirige la Fonderie Brun, ou il a fait toute sa carriere.
Paul est le pere de Nadia, et le grand-pere de Lucas.
Sophie est nee a Lyon.
Un vieux registre pretend pourtant que Sophie serait nee a Bordeaux.
Une note marginale affirme que la Fonderie Brun a ete fondee par l'Atelier Verne.
""".strip()

# Schema : vocabulaire ferme de relations, avec signatures typees
PERSONS = {"henri", "claire", "marc", "sophie", "lucas", "nadia", "paul"}
ORGS = {"atelier_verne", "fonderie_brun"}
PLACES = {"lyon", "bordeaux"}
TYPE_OF = {**{e: "personne" for e in PERSONS},
           **{e: "organisation" for e in ORGS},
           **{e: "lieu" for e in PLACES}}

SIGNATURES = {
    "parent":         ("personne", "personne"),
    "grandparent":    ("personne", "personne"),
    "dirige":         ("personne", "organisation"),
    "travaille_pour": ("personne", "organisation"),
    "fondateur_de":   ("personne", "organisation"),
    "ne_a":           ("personne", "lieu"),
}
# Contraintes fonctionnelles : (relation, position fixee) -> l'autre est unique
FUNCTIONAL = {("ne_a", "sujet"),    # une personne a UN lieu de naissance
              ("dirige", "objet")}  # une organisation a UN dirigeant

print(f"Corpus : {len(CORPUS.splitlines())} enonces")
print(f"Schema : {len(SIGNATURES)} relations typees, "
      f"{len(PERSONS)} personnes, {len(ORGS)} organisations, {len(PLACES)} lieux")
print(f"Contraintes fonctionnelles : {sorted(FUNCTIONAL)}")

Corpus : 13 enonces
Schema : 6 relations typees, 7 personnes, 2 organisations, 2 lieux
Contraintes fonctionnelles : [('dirige', 'objet'), ('ne_a', 'sujet')]


Le schema est notre **biais de langage** (SL-4, SL-9) : un vocabulaire ferme de
relations, des types, des contraintes. C'est lui qui rend l'oracle possible ---
et c'est lui qui decidera aussi de ce que le pipeline est *incapable* de voir.


---

## 3. Etage 1 : extraction par LLM

Meme dispositif qu'en SL-7 : un `.env` a la racine de la serie configure
n'importe quel endpoint OpenAI-compatible (ici OpenRouter / Gemini 3.5 Flash) ;
sans `.env`, un simulateur deterministe --- l'extraction de reference --- prend
le relais pour que le notebook reste executable hors-ligne.


In [2]:
# Configuration de l'acces LLM via .env (endpoint OpenAI-compatible)
import os
import re
from dotenv import load_dotenv

load_dotenv()  # charge le .env du repertoire du notebook s'il existe

LLM_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
LLM_MODEL = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-4o-mini")
LLM_AVAILABLE = bool(LLM_API_KEY)

if LLM_AVAILABLE:
    from openai import OpenAI
    llm_client = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
    print("LLM reel configure :")
    print(f"  Endpoint : {LLM_BASE_URL}")
    print(f"  Modele   : {LLM_MODEL}")
    print("  (la cle API n'est jamais affichee)")
else:
    llm_client = None
    print("Pas de .env / OPENAI_API_KEY : extraction par le simulateur de reference.")

LLM reel configure :
  Endpoint : https://openrouter.ai/api/v1
  Modele   : google/gemini-3.5-flash
  (la cle API n'est jamais affichee)


In [3]:
# Extraction de triplets : prompt structure + parsing + filet de securite

# L'extraction de reference (= la verite terrain, et le simulateur hors-ligne).
# Les DEUX derniers triplets sont les pieges du corpus, extraits fidelement :
# l'extraction rapporte ce que le texte AFFIRME, l'oracle jugera.
GOLD_TRIPLES = [
    ("fondateur_de", "henri", "atelier_verne"),
    ("parent", "henri", "claire"), ("parent", "henri", "marc"),
    ("dirige", "claire", "atelier_verne"), ("travaille_pour", "claire", "atelier_verne"),
    ("travaille_pour", "marc", "atelier_verne"),
    ("parent", "claire", "sophie"), ("travaille_pour", "sophie", "atelier_verne"),
    ("parent", "marc", "lucas"),
    ("grandparent", "henri", "lucas"),
    ("parent", "nadia", "lucas"), ("travaille_pour", "nadia", "fonderie_brun"),
    ("dirige", "paul", "fonderie_brun"), ("travaille_pour", "paul", "fonderie_brun"),
    ("parent", "paul", "nadia"), ("grandparent", "paul", "lucas"),
    ("ne_a", "sophie", "lyon"),
    ("ne_a", "sophie", "bordeaux"),                  # piege 1 : registre contradictoire
    ("fondateur_de", "atelier_verne", "fonderie_brun"),  # piege 2 : fondateur non humain
]

ENTITY_IDS = sorted(PERSONS | ORGS | PLACES)
EXTRACTION_PROMPT = f"""Extrais TOUS les faits affirmes par le texte ci-dessous sous forme
de triplets, un par ligne, au format strict : relation(sujet, objet)

Relations autorisees (aucune autre) : {", ".join(sorted(SIGNATURES))}
- parent(X, Y) signifie : X est le pere ou la mere de Y
- grandparent(X, Y) signifie : X est le grand-pere ou la grand-mere de Y
Identifiants d'entites a utiliser EXACTEMENT : {", ".join(ENTITY_IDS)}

Extrais aussi les faits presentes comme rumeurs ou notes douteuses : le filtrage
viendra apres. Reponds UNIQUEMENT avec les triplets, sans commentaire.

Texte :
{CORPUS}"""

def parse_triples(text):
    triples = []
    for m in re.finditer(r"(\w+)\(\s*([\w_]+)\s*,\s*([\w_]+)\s*\)", text):
        r, s, o = m.group(1).lower(), m.group(2).lower(), m.group(3).lower()
        if r in SIGNATURES and (r, s, o) not in triples:
            triples.append((r, s, o))
    return triples

if LLM_AVAILABLE:
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": EXTRACTION_PROMPT}],
        temperature=0,
        max_tokens=4000,  # les modeles 'reasoning' consomment des tokens AVANT la reponse
    )
    llm_triples = parse_triples(response.choices[0].message.content or "")
    source = LLM_MODEL
else:
    llm_triples = list(GOLD_TRIPLES)
    source = "simulateur (extraction de reference)"

# Qualite de l'extraction vs la reference
gold, got = set(GOLD_TRIPLES), set(llm_triples)
recall = len(gold & got) / len(gold)
precision = len(gold & got) / len(got) if got else 0.0
print(f"Extraction ({source}) : {len(llm_triples)} triplets")
print(f"  Rappel    : {len(gold & got)}/{len(gold)} = {recall:.2f}")
print(f"  Precision : {len(gold & got)}/{len(got)} = {precision:.2f}")
extras = sorted(got - gold)
if extras:
    print("  Triplets hors reference : " + ", ".join(f"{r}({s}, {o})" for r, s, o in extras))

# Filet de securite pour la suite du pipeline : on complete avec les faits
# manques (en pratique : extraction multi-passes + vote, cf le defi de l'Ex. 1)
missed = sorted(gold - got)
if missed:
    print(f"  Faits manques reinjectes pour la suite : "
          + ", ".join(f"{r}({s}, {o})" for r, s, o in missed))
triples = llm_triples + [t for t in GOLD_TRIPLES if t not in got]
print(f"\nEntree de l'oracle : {len(triples)} triplets")

Extraction (google/gemini-3.5-flash) : 19 triplets
  Rappel    : 19/19 = 1.00
  Precision : 19/19 = 1.00

Entree de l'oracle : 19 triplets


### Interpretation

L'etage neuronal fait ce qu'aucune regex ne sait faire : lire « Nadia, la mere de
Lucas » et en tirer `parent(nadia, lucas)`. Mais il reste *faillible et non
verifie* : le rappel et la precision affiches ci-dessus sont mesurables uniquement
parce que nous possedons une extraction de reference --- luxe inexistant en
production. (Sur ce corpus jouet --- 13 enonces, vocabulaire
ferme, identifiants imposes --- Gemini 3.5 Flash realise une extraction
parfaite ; rien ne garantit qu'elle le reste sur du texte ouvert.) D'ou l'etage suivant : tout triplet, d'ou qu'il vienne, passe devant
l'oracle. Notez que l'extraction rapporte fidelement les deux pieges du corpus :
c'est le *role* de l'extraction de rester fidele au texte, et celui de l'oracle
de juger.


---

## 4. Etage 2 : l'oracle symbolique

Trois verifications, dans l'esprit de la consistance du Version Space (SL-1) et
des determinations (SL-3) : **entites connues**, **types conformes a la
signature**, **contraintes fonctionnelles** (une personne n'a qu'un lieu de
naissance ; une organisation n'a qu'un dirigeant). Politique de conflit :
premier-arrive, premier-cru --- politique discutable, et c'est un exercice.


In [4]:
def oracle(triples):
    accepted, rejected = [], []
    for (r, s, o) in triples:
        sig = SIGNATURES.get(r)
        if sig is None:
            rejected.append(((r, s, o), "relation inconnue")); continue
        if s not in TYPE_OF or o not in TYPE_OF:
            rejected.append(((r, s, o), "entite inconnue")); continue
        if (TYPE_OF[s], TYPE_OF[o]) != sig:
            rejected.append(((r, s, o),
                f"types ({TYPE_OF[s]}, {TYPE_OF[o]}) != signature {sig}")); continue
        conflict = None
        if (r, "sujet") in FUNCTIONAL:
            conflict = next((t for t in accepted if t[0] == r and t[1] == s and t[2] != o), None)
        if (r, "objet") in FUNCTIONAL and conflict is None:
            conflict = next((t for t in accepted if t[0] == r and t[2] == o and t[1] != s), None)
        if conflict:
            rejected.append(((r, s, o),
                f"contrainte fonctionnelle : {conflict[0]}({conflict[1]}, {conflict[2]}) deja admis"))
            continue
        accepted.append((r, s, o))
    return accepted, rejected


kg_facts, rejected = oracle(triples)
print(f"Oracle : {len(kg_facts)} acceptes, {len(rejected)} rejetes\n")
print("Rejets :")
for (r, s, o), reason in rejected:
    print(f"  {r}({s}, {o})")
    print(f"      -> {reason}")

Oracle : 17 acceptes, 2 rejetes

Rejets :
  ne_a(sophie, bordeaux)
      -> contrainte fonctionnelle : ne_a(sophie, lyon) deja admis
  fondateur_de(atelier_verne, fonderie_brun)
      -> types (organisation, organisation) != signature ('personne', 'organisation')


### Interpretation

Les deux pieges sont morts ici, chacun tue par une arme differente :

- `fondateur_de(atelier_verne, fonderie_brun)` --- la note marginale --- viole la
  **signature** : un fondateur est une personne dans notre schema. Verification
  purement structurelle, zero connaissance du monde.
- `ne_a(sophie, bordeaux)` --- le vieux registre --- viole la **contrainte
  fonctionnelle** : Lyon est arrive avant. Remarquez l'arbitraire : l'oracle n'a
  aucun moyen de savoir *lequel* des deux registres ment, il applique une
  politique d'anciennete. Truth discovery, gestion des sources, votes ponderes :
  tout un champ de recherche vit dans cette ligne (defi de l'Ex. 2).

Un LLM interroge sans cet etage aurait pu retenir Bordeaux, Lyon, ou les deux
selon le prompt. L'oracle, lui, est *previsible* --- y compris dans ses erreurs.


---

## 5. Etage 3 : le graphe de connaissances

Les faits valides forment un KG (SL-6) : sommets = entites, aretes etiquetees =
relations. C'est le substrat sur lequel les deux etages suivants vont travailler.


In [5]:
from collections import defaultdict

by_rel = defaultdict(list)
for r, s, o in kg_facts:
    by_rel[r].append((s, o))

print(f"KG : {len(kg_facts)} faits, {len(by_rel)} relations, "
      f"{len({e for _, s, o in kg_facts for e in (s, o)})} entites\n")
for r in sorted(by_rel):
    pairs = ", ".join(f"({s}, {o})" for s, o in sorted(by_rel[r]))
    print(f"  {r:<15} {pairs}")

KG : 17 faits, 6 relations, 10 entites

  dirige          (claire, atelier_verne), (paul, fonderie_brun)
  fondateur_de    (henri, atelier_verne)
  grandparent     (henri, lucas), (paul, lucas)
  ne_a            (sophie, lyon)
  parent          (claire, sophie), (henri, claire), (henri, marc), (marc, lucas), (nadia, lucas), (paul, nadia)
  travaille_pour  (claire, atelier_verne), (marc, atelier_verne), (nadia, fonderie_brun), (paul, fonderie_brun), (sophie, atelier_verne)


---

## 6. Etage 4 : miner des regles (AMIE-lite)

Le mineur enumere deux gabarits de regles de Horn --- l'implication directe
$h(X,Y) \leftarrow b(X,Y)$ et la chaine $h(X,Y) \leftarrow b_1(X,Z) \wedge
b_2(Z,Y)$ --- et les score comme AMIE (SL-6) :

$$\text{support} = |\{(x,y) : \text{corps}(x,y) \wedge h(x,y)\}| \qquad
\text{confiance} = \frac{\text{support}}{|\{(x,y) : \text{corps}(x,y)\}|}$$

Seuils : support $\geq 2$, confiance $\geq 0.6$. La grammaire est minuscule
comparee a SL-9 (pas de clause bottom ici) mais l'esprit est le meme : un espace
d'hypotheses borne, un score, un seuil.


In [6]:
def mine_rules(facts, min_support=2, min_conf=0.6):
    rels = defaultdict(set)
    for r, s, o in facts:
        rels[r].add((s, o))
    names = sorted(rels)
    candidates = []
    for h in names:                       # gabarit 1 : h(X,Y) :- b(X,Y)
        for b in names:
            if h == b:
                continue
            inst = rels[b]
            support = len(inst & rels[h])
            if support >= min_support:
                candidates.append({"kind": "impl", "head": h, "b1": b, "b2": None,
                                   "support": support, "conf": support / len(inst),
                                   "text": f"{h}(X,Y) :- {b}(X,Y)"})
    for h in names:                       # gabarit 2 : h(X,Y) :- b1(X,Z), b2(Z,Y)
        for b1 in names:
            for b2 in names:
                inst = {(x, y) for (x, z) in rels[b1]
                        for (z2, y) in rels[b2] if z == z2 and x != y}
                if not inst:
                    continue
                support = len(inst & rels[h])
                if support >= min_support:
                    candidates.append({"kind": "chain", "head": h, "b1": b1, "b2": b2,
                                       "support": support, "conf": support / len(inst),
                                       "text": f"{h}(X,Y) :- {b1}(X,Z), {b2}(Z,Y)"})
    return candidates, min_conf


candidates, MIN_CONF = mine_rules(kg_facts)
print(f"{'regle':<55} | support | conf | verdict")
print("-" * 85)
for c in sorted(candidates, key=lambda c: -c["conf"]):
    verdict = "RETENUE" if c["conf"] >= MIN_CONF else "rejetee"
    print(f"{c['text']:<55} |    {c['support']}    | {c['conf']:.2f} | {verdict}")
rules = [c for c in candidates if c["conf"] >= MIN_CONF]

regle                                                   | support | conf | verdict
-------------------------------------------------------------------------------------
travaille_pour(X,Y) :- dirige(X,Y)                      |    2    | 1.00 | RETENUE
dirige(X,Y) :- parent(X,Z), travaille_pour(Z,Y)         |    2    | 0.67 | RETENUE
grandparent(X,Y) :- parent(X,Z), parent(Z,Y)            |    2    | 0.67 | RETENUE
travaille_pour(X,Y) :- parent(X,Z), travaille_pour(Z,Y) |    2    | 0.67 | RETENUE
dirige(X,Y) :- travaille_pour(X,Y)                      |    2    | 0.40 | rejetee


### Interpretation : quatre retenues, dont deux suspectes

- `travaille_pour(X,Y) :- dirige(X,Y)` (confiance 1.00) : du quasi-schema --- un
  dirigeant travaille pour son organisation. Vrai, mais peu informatif.
- `grandparent(X,Y) :- parent(X,Z), parent(Z,Y)` (confiance 0.67) : **la**
  definition, retrouvee par comptage pur. La confiance n'est pas 1.0 car le KG
  connait un couple parent-de-parent (Henri-Claire-Sophie) sans fait
  `grandparent` correspondant --- c'est exactement le trou que l'inference va
  combler.
- `travaille_pour(X,Y) :- parent(X,Z), travaille_pour(Z,Y)` (confiance 0.67) :
  premiere regle *suspecte*. Dans notre saga d'entreprises familiales, les
  parents travaillent souvent la ou travaillent leurs enfants --- correlation de
  ce micro-monde, pas loi du monde reel.
- `dirige(X,Y) :- parent(X,Z), travaille_pour(Z,Y)` (confiance 0.67) : la
  franchement fausse --- « on dirige l'entreprise ou travaillent ses enfants » !
  Elle doit sa confiance aux memes coincidences (Claire dirige la ou travaille
  sa fille Sophie, Paul la ou travaille sa fille Nadia). Le mineur ne fait pas la
  difference : **support et confiance mesurent la regularite des donnees, pas la
  causalite**.
- `dirige(X,Y) :- travaille_pour(X,Y)` (confiance 0.40) tombe sous le seuil...
  alors qu'elle est moins absurde que la precedente. Le seuil filtre par
  frequence, pas par sens --- l'exercice 3 quantifie cette impuissance.


---

## 7. Etage 5 : chainage avant --- la decouverte

On applique les regles retenues jusqu'au point fixe (chainage avant, SL-2), en
gardant pour chaque fait nouveau sa **provenance** : la regle et les faits
temoins qui l'ont produit.


In [7]:
def forward_chain(facts, rules):
    known = set(facts)
    derived = []
    changed = True
    while changed:
        changed = False
        rels = defaultdict(set)
        for r, s, o in known:
            rels[r].add((s, o))
        for rule in rules:
            if rule["kind"] == "impl":
                produced = [((rule["head"], s, o), [(rule["b1"], s, o)])
                            for (s, o) in rels[rule["b1"]]]
            else:
                produced = [((rule["head"], x, y),
                             [(rule["b1"], x, z), (rule["b2"], z, y)])
                            for (x, z) in rels[rule["b1"]]
                            for (z2, y) in rels[rule["b2"]] if z == z2 and x != y]
            for fact, witnesses in produced:
                if fact not in known:
                    known.add(fact)
                    derived.append((fact, rule, witnesses))
                    changed = True
    return known, derived


kg_inferred, derived = forward_chain(kg_facts, rules)
print(f"Chainage avant : {len(derived)} fait(s) nouveau(x) "
      f"(KG : {len(kg_facts)} -> {len(kg_inferred)})\n")
for (r, s, o), rule, witnesses in derived:
    print(f"  NOUVEAU : {r}({s}, {o})")
    print(f"    regle   : {rule['text']}  (conf {rule['conf']:.2f})")
    print(f"    temoins : " + ", ".join(f"{w[0]}({w[1]}, {w[2]})" for w in witnesses))

Chainage avant : 3 fait(s) nouveau(x) (KG : 17 -> 20)

  NOUVEAU : dirige(henri, atelier_verne)
    regle   : dirige(X,Y) :- parent(X,Z), travaille_pour(Z,Y)  (conf 0.67)
    temoins : parent(henri, marc), travaille_pour(marc, atelier_verne)
  NOUVEAU : grandparent(henri, sophie)
    regle   : grandparent(X,Y) :- parent(X,Z), parent(Z,Y)  (conf 0.67)
    temoins : parent(henri, claire), parent(claire, sophie)
  NOUVEAU : travaille_pour(henri, atelier_verne)
    regle   : travaille_pour(X,Y) :- parent(X,Z), travaille_pour(Z,Y)  (conf 0.67)
    temoins : parent(henri, marc), travaille_pour(marc, atelier_verne)


### Interpretation : le payoff... et les poisons

Trois faits nouveaux, trois statuts :

1. **`grandparent(henri, sophie)`** --- la decouverte. *Aucune phrase du corpus
   ne le dit.* Le pipeline l'a obtenu en combinant deux faits extraits par le LLM
   et une regle minee par comptage : c'est la promesse neuro-symbolique tenue, et
   la provenance affichee en est la *preuve auditable*.

2. **`travaille_pour(henri, atelier_verne)`** --- l'artefact plausible. Produit
   par la premiere regle suspecte : le fondateur a sans doute travaille dans son
   atelier... avant la retraite ? Rien dans le texte ne le soutient.

3. **`dirige(henri, atelier_verne)`** --- l'artefact *illegal*. Non seulement
   probablement faux (c'est Claire qui dirige), mais il **viole la contrainte
   fonctionnelle que l'oracle a fait respecter a l'etage 2** : une organisation,
   un dirigeant. Le KG contient desormais deux dirigeants pour l'Atelier Verne.
   La lecon est architecturale : **le chainage avant injecte ses conclusions
   sans les re-soumettre a l'oracle**. Un pipeline coherent re-validerait chaque
   fait derive --- et un rejet devrait alors faire baisser la confiance de la
   regle fautive (boucle de retro-action que le defi de l'Ex. 4 explore).

Meme mecanisme, meme confiance 0.67, trois valeurs de verite differentes. La
seule difference visible de l'exterieur : la **provenance**, qui permet au moins
de remonter la chaine et d'incriminer la regle.


---

## 8. Etage 6 : la boucle bouclee --- LLM seul contre KG raisonne

Question multi-sauts dont la reponse complete exige l'inference : « Qui sont les
petits-enfants de Henri ? » Le corpus ne nomme que Lucas ; Sophie n'est
atteignable que par raisonnement. On compare la reponse du LLM lisant le corpus
a celle du KG enrichi.


In [8]:
QUESTION = "Qui sont les petits-enfants de Henri ? Reponds en une phrase."

# Reponse du KG : simple requete sur le graphe enrichi
kg_answer = sorted(o for (r, s, o) in kg_inferred if r == "grandparent" and s == "henri")
print(f"Reponse du KG raisonne : {kg_answer}")
for (r, s, o), rule, witnesses in derived:
    if r == "grandparent" and s == "henri":
        print(f"  dont {o} INFERE via {rule['text']}")
        print(f"  temoins : " + ", ".join(f"{w[0]}({w[1]}, {w[2]})" for w in witnesses))

# Reponse du LLM seul, corpus en contexte
if LLM_AVAILABLE:
    qa = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user",
                   "content": f"Texte :\n{CORPUS}\n\nQuestion : {QUESTION}"}],
        temperature=0,
        max_tokens=4000,
    )
    print(f"\nReponse du LLM seul ({LLM_MODEL}) :")
    print(f"  {(qa.choices[0].message.content or '').strip()}")
else:
    print("\n(pas de .env : comparaison LLM indisponible hors-ligne)")

Reponse du KG raisonne : ['lucas', 'sophie']
  dont sophie INFERE via grandparent(X,Y) :- parent(X,Z), parent(Z,Y)
  temoins : parent(henri, claire), parent(claire, sophie)



Reponse du LLM seul (google/gemini-3.5-flash) :
  Les petits-enfants de Henri sont Sophie et Lucas.


### Interpretation

Que le LLM reussisse ou non ce saut a deux pas (les modeles recents y arrivent
souvent), la difference de *nature* entre les deux reponses demeure :

- la reponse du KG vient avec une **derivation** : deux faits temoins, une regle,
  sa confiance --- chaque maillon est inspectable, contestable, corrigeable ;
- la reponse du LLM est une **affirmation** : peut-etre juste, mais sans autre
  garantie que la vraisemblance statistique, et silencieusement dependante du
  modele, du prompt et de la temperature (SL-7).

C'est le partage des roles defendu depuis SL-5 : au neuronal la *lecture*, au
symbolique la *responsabilite*. Et l'angle mort reste partage lui aussi : le KG
affirme `travaille_pour(henri, atelier_verne)` --- et meme
`dirige(henri, atelier_verne)`, en contradiction avec sa propre contrainte
fonctionnelle --- avec le meme aplomb --- la
provenance est une piste d'audit, pas un certificat de verite.


---

## 9. Exercices

### Tableau recapitulatif

| Etage | Mecanisme | Fonction |
|-------|-----------|----------|
| Extraction | LLM + vocabulaire ferme + parsing | texte -> triplets candidats |
| Oracle | types, signatures, contraintes fonctionnelles | triplets -> faits valides |
| KG | index par relation | stockage + requetes |
| Mining | gabarits impl/chaine, support + confiance | faits -> regles scorees |
| Inference | chainage avant + provenance | regles -> faits nouveaux |
| Confrontation | requete KG vs question LLM | audit de la reponse |


In [9]:
# Exercice 1 : Etendre le schema
# TODO etudiant : ajoutez la relation marie_avec (personne, personne) au schema,
# deux phrases au corpus (ex : "Henri est marie a Jeanne."), l'entite jeanne,
# puis re-executez extraction + oracle. La relation est SYMETRIQUE : ajoutez a
# l'oracle la completion automatique marie_avec(Y,X) des que marie_avec(X,Y)
# est admis.
# Indice : SIGNATURES, PERSONS, CORPUS, et la boucle de oracle() sont a etendre.
# Etape 1 : schema + corpus etendus
# Etape 2 : re-extraction (LLM ou simulateur) et oracle
# Etape 3 : verifiez que les deux sens du mariage sont dans le KG

print("Exercice a completer")

Exercice a completer


L'exercice suivant attaque la politique de conflit de l'oracle --- le
premier-arrive-premier-cru applique au registre de Bordeaux.


In [10]:
# Exercice 2 : Une meilleure politique de conflit
# TODO etudiant : remplacez le premier-arrive-premier-cru par une politique a
# sources : chaque triplet arrive maintenant avec une etiquette de source
# ("texte_principal" ou "vieux_registre") et un score de fiabilite. En cas de
# conflit fonctionnel, gardez le triplet de la source la plus fiable, QUEL QUE
# SOIT l'ordre d'arrivee.
# Indice : transformez les triplets en (relation, sujet, objet, source) et
# donnez fiabilite 0.9 au texte, 0.4 au registre. Testez les DEUX ordres.
# Etape 1 : oracle_avec_sources(triples_etiquetes, fiabilite)
# Etape 2 : verifiez que lyon gagne meme si bordeaux arrive en premier
# Etape 3 : que se passe-t-il si deux sources ont la meme fiabilite ?

print("Exercice a completer")

Exercice a completer


L'exercice suivant quantifie ce que les seuils du mineur font vraiment :
chaque choix de seuil est un compromis entre regles manquees et regles fausses.


In [11]:
# Exercice 3 : Le bon seuil n'existe pas
# TODO etudiant : faites varier min_conf de 0.0 a 1.0 (pas de 0.1) et classez a
# la main chaque regle candidate comme VRAIE (loi du domaine) ou FAUSSE
# (correlation du micro-monde). Tracez (ou tabulez) le nombre de vraies regles
# retenues et de fausses regles retenues a chaque seuil.
# Indice : grandparent :- parent o parent est VRAIE ; travaille_pour :- parent o
# travaille_pour est FAUSSE ; les deux ont la MEME confiance 0.67...
# Etape 1 : tableau seuil -> (vraies retenues, fausses retenues)
# Etape 2 : existe-t-il un seuil qui garde toutes les vraies et aucune fausse ?
# Etape 3 : que faudrait-il changer (donnees ? gabarits ? score ?) pour separer
#           ces deux regles ?

print("Exercice a completer")

Exercice a completer


Dernier exercice : l'attaque de bout en bout. Une phrase fausse mais
parfaitement typee traverse-t-elle tout le pipeline ?


In [12]:
# Exercice 4 : Empoisonnement de bout en bout
# TODO etudiant : ajoutez au corpus la phrase fausse mais bien typee
# "Lucas est le pere de Sophie." et re-executez TOUT le pipeline (extraction,
# oracle, mining, inference). Tracez la propagation : quels faits nouveaux
# (faux) apparaissent par chainage avant ? Quelle etape aurait PU l'arreter ?
# Indice : parent(lucas, sophie) passe l'oracle (types corrects, pas de
# contrainte fonctionnelle sur parent) puis nourrit la regle grandparent...
# Etape 1 : pipeline complet avec la phrase empoisonnee
# Etape 2 : listez les faits derives contamines et leur provenance
# Etape 3 : proposez (en commentaire) la contrainte qui l'aurait bloquee,
#           et ce qu'elle exigerait du schema (ages ? generations ? cycles ?)

print("Exercice a completer")

Exercice a completer


---

## Defi presentation (seance du 17 juin)

Modalite du cours : chaque groupe choisit **un exercice** de la serie, le prepare,
et le presente en seance. Resoudre l'exercice est le minimum ; ce qui distingue une
presentation qui maitrise le sujet, c'est la **question-twist** associee ci-dessous.
Elle fait partie integrante de la presentation attendue.

| Exercice | Question-twist a traiter en plus |
|----------|----------------------------------|
| **Ex. 1 (etendre le schema)** | Le mineur va decouvrir `marie_avec(X,Y) :- marie_avec(Y,X)` avec confiance 1.0 : une regle qui ne fait que refleter votre propre completion d'oracle. Comment distinguer mecaniquement une regle *apprise des donnees* d'une regle qui reapprend le *schema* qu'on y a injecte ? |
| **Ex. 2 (politique de conflit)** | Votre politique a sources suppose les fiabilites connues a priori. En *truth discovery*, on les estime en meme temps que les faits (une source est fiable si elle affirme des faits confirmes...). Esquissez l'algorithme de point fixe correspondant et son cas pathologique. |
| **Ex. 3 (seuils)** | `grandparent :- parent o parent` (vraie) et `travaille_pour :- parent o travaille_pour` (fausse) ont la meme confiance 0.67 : aucun seuil ne les separe. Quelle information NON presente dans le KG les separerait (contre-exemples ? intervention ? texte ?) --- et lequel des dix notebooks de la serie fournit l'outil le plus proche ? |
| **Ex. 4 (empoisonnement)** | Classez les defenses par etage (modalite epistemique a l'extraction, contraintes temporelles a l'oracle, pedigree a l'inference) et defendez la these : « la provenance est la seule defense qui passe a l'echelle ». Quelle requete d'apprentissage *actif* (SL-8) le pipeline devrait-il poser pour trancher `travaille_pour(henri, atelier_verne)` ? |


---

## 10. Conclusion

### Ce que le pipeline demontre

1. **La division du travail neuro-symbolique fonctionne** : le LLM a lu le texte,
   l'oracle a tue les deux pieges, le mineur a retrouve la definition du
   grand-parent, le chainage a decouvert un fait que personne n'avait ecrit ---
   avec sa preuve.

2. **Chaque etage a son angle mort** : l'extraction depend du schema et du
   modele ; l'oracle tranche les conflits par anciennete ; le mineur confond
   regularite et causalite ; l'inference propage les artefacts avec la meme
   assurance que les decouvertes --- et re-injecte ses conclusions sans les
   re-soumettre a l'oracle (deux dirigeants pour l'Atelier Verne !). Le pipeline n'est pas une forteresse, c'est une
   *chaine de responsabilites* --- et la provenance est son registre.

3. **La serie en un tableau** :

| Etage du pipeline | Outil | Herite de |
|-------------------|-------|-----------|
| Schema / biais de langage | vocabulaire ferme, signatures | [SL-4](SL-4-InductiveLogicProgramming.ipynb), [SL-9](SL-9-InverseResolution.ipynb) |
| Extraction | LLM + parsing + filets | [SL-7](SL-7-LLM-SymbolicLearning.ipynb) |
| Oracle de consistance | types, contraintes, conflits | [SL-1](SL-1-LogicalLearning.ipynb), [SL-3](SL-3-RelevanceLearning.ipynb) |
| Graphe de connaissances | index relationnel | [SL-6](SL-6-KnowledgeGraphs-ILP.ipynb) |
| Mining de regles | support, confiance, seuils | [SL-6](SL-6-KnowledgeGraphs-ILP.ipynb), [SL-9](SL-9-InverseResolution.ipynb) |
| Chainage + provenance | regles compilees, forward | [SL-2](SL-2-KnowledgeBasedLearning.ipynb) |
| Hybridation, audit | LLM vs raisonneur | [SL-5](SL-5-NeuroSymbolic.ipynb), [SL-7](SL-7-LLM-SymbolicLearning.ipynb) |
| (manquant : requetes actives) | poser LA bonne question | [SL-8](SL-8-ActiveAutomataLearning.ipynb), defi Ex. 4 |

### Pour aller plus loin

Le pipeline jouet de ce notebook a des cousins industriels : les systemes
d'extraction d'information neuronale + validation par schema (NELL, Google
Knowledge Vault), les mineurs de regles sur KG massifs (AMIE+ sur Wikidata), et
la generation augmentee par graphe (GraphRAG). Les ingredients sont les memes ;
seules les echelles changent.


---

## Ressources

- A. Carlson et al., *Toward an Architecture for Never-Ending Language Learning* (NELL), AAAI 2010
- X. Dong et al., *Knowledge Vault: A Web-Scale Approach to Probabilistic Knowledge Fusion*, KDD 2014
- L. Galarraga et al., *Fast Rule Mining in Ontological Knowledge Bases with AMIE+*, VLDB Journal 2015
- Y. Li et al., *Truth Discovery with Multiple Conflicting Information Providers*, TKDE 2008
- D. Edge et al., *From Local to Global: A GraphRAG Approach to Query-Focused Summarization*, 2024
- Russell & Norvig, *AI: A Modern Approach*, 3e ed., chap. 19 (le fil de toute la serie)

---

**Retour** : [Index SymbolicLearning](./README.md) | [<< SL-9](SL-9-InverseResolution.ipynb)
